# Model setup + Lasso & XGBoost models

Shared train/test split, rolling-origin CV folds, and evaluation metrics,
now combined with two working models (Lasso and XGBoost) so the whole
pipeline runs end-to-end in one notebook.

**Fixes applied vs. the original shared setup file:**
- `FEATURE_COLS` now excludes raw balance-sheet levels (`toas`, `ncli`,
  `culi`, `shfd`, ...). Those are collinear with each other (accounting
  identities: `toas == tshf`, `ncli == ltdb + oncl`, etc.) and contain
  extreme outliers (`toas` up to ~7.9e14 in the real data) — feeding them
  into unregularized/linear models makes coefficients unstable. The
  engineered ratios (`leverage`, `solvency`, ...) and `log_toas`/`log_empl`
  are used instead.
- `train_df` / `test_df` are explicitly `.copy()`'d right after the split
  to silence `SettingWithCopyWarning` and avoid mutating a view.
- Numeric columns are downcast to `float32` and each fold loop explicitly
  frees fitted objects (`del` + `gc.collect()`) to reduce peak memory —
  this is what caused the `MemoryError` in the original notebook (it died
  on fold 3, ~520K rows, after folds 1–2 succeeded).

## 1. Load data

In [1]:
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA_PATH = "data/model_input/analysis_panel.parquet"  # <- adjust to your actual path

# Optional: set a fraction (e.g. 0.1) to work on a random sample while
# testing/iterating, then set back to None for the real full run.
SAMPLE_FRAC = None

# --- Memory fix 1: skip columns we don't need, WITHOUT loading the file twice ---
# 'name' is never used as a feature. 'dateinc' isn't needed either --
# 'firm_age' is already precomputed as its own column. 'report_date' is
# redundant with 'closdate_year', which we use directly as the year column.
# Reading just the schema (not the data) via pyarrow is essentially free,
# unlike pd.read_parquet(...).columns which would read the whole file first.
DROP_COLS_AT_LOAD = ["name", "dateinc", "report_date"]

schema_cols = pq.ParquetFile(DATA_PATH).schema_arrow.names
usecols = [c for c in schema_cols if c not in DROP_COLS_AT_LOAD]

df = pd.read_parquet(DATA_PATH, columns=usecols)

if SAMPLE_FRAC:
    df = df.sample(frac=SAMPLE_FRAC, random_state=0).reset_index(drop=True)
    print(f"(dev mode: sampled {SAMPLE_FRAC:.0%} of rows)")

# --- Memory fix 2: downcast numeric columns to float32 right away ---
float_cols = df.select_dtypes(include="float64").columns
df[float_cols] = df[float_cols].astype("float32")
gc.collect()

print(df.shape)
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

(1745282, 77)
Memory usage: 0.69 GB


,lm_positive,lm_negative,lm_polarity,uncertainty_ratio,litigious_ratio,constraining_ratio,strong_modal_ratio,weak_modal_ratio,ifo_business_climate,ifo_business_situation,...,log_toas,log_empl,ncliGrowthThisYear,toas_growth,cash_growth,growth_volatility,firm_age,years_in_panel,naics_2digit,is_na_empl
0,265,402,-0.205397,0.020557,0.014647,0.003769,0.001970,0.006167,98.199997,98.400002,...,15.700995,4.110874,NaN,NaN,NaN,NaN,24.0,0,42,0
1,301,447,-0.195187,0.018509,0.033679,0.014682,0.001600,0.004593,100.699997,100.599998,...,15.678370,3.970292,0.384772,-0.022372,0.473965,NaN,25.0,1,42,0
2,352,677,-0.315841,0.021990,0.012566,0.005463,0.002117,0.005600,101.000000,101.800003,...,15.772503,3.912023,-0.401733,0.098706,0.018104,0.556143,26.0,2,42,0
3,397,547,-0.158898,0.016656,0.015763,0.006186,0.002141,0.004342,104.699997,107.099998,...,15.757667,3.912023,0.816768,-0.014727,0.616062,0.617786,27.0,3,42,0
4,359,513,-0.176606,0.020424,0.012030,0.006295,0.002518,0.004337,101.199997,105.300003,...,15.728341,3.931826,-0.247863,-0.028900,-0.008948,0.663559,28.0,4,42,0


In [2]:
# We already have 'closdate_year' as a clean int column -- no need to
# parse 'report_date' into a year separately (removed: this used to load
# report_date as a datetime column and re-derive the same information,
# doubling work and memory for no benefit).
YEAR_COL_SOURCE = "closdate_year"  # used directly in Section 2 below

## 1b. New engineered features

Two additions discussed alongside the existing ratio set:

- **`fixed_assets_share = fias / toas`** — asset composition (capital-intensive
  vs. asset-light firms). Complements `inventory_share` / `receivables_share`
  / `cash_ratio`, which only cover the *current assets* side; this covers the
  long-term side (`fias + cuas = toas`, so this is `1 - current_assets_share`).
- **`ltdb_share_of_ncli = ltdb / ncli`** — within a firm's existing non-current
  liabilities, how much is actual long-term *debt* (`ltdb`) vs. provisions /
  other obligations (`oncl`, e.g. pensions). The outcome variable
  (`ncliGrowthNextYear`) is growth in the *total*, which mixes these two very
  different things together — this ratio gives the model a signal for which
  one a firm's liabilities are currently made of.

Both use only period-*t* raw levels (already loaded, just excluded from
`FEATURE_COLS` directly) — no leakage risk, same as the rest of the ratio set.

**Note on winsorizing:** for consistency with how the rest of the ratio set
was built upstream, bounds are computed on the whole dataset here (not
train-only). If you want to be stricter about avoiding any train/test
information mixing, compute these bounds on `train_df` only after the split
in Section 4 and apply them to `test_df` — worth doing consistently across
*all* ratio features if you go that route, not just these two.

In [3]:
def winsorize(s, lower=0.01, upper=0.99):
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)

df["fixed_assets_share"] = df["fias"] / df["toas"]
df["ltdb_share_of_ncli"] = df["ltdb"] / df["ncli"]

new_ratio_cols = ["fixed_assets_share", "ltdb_share_of_ncli"]
df[new_ratio_cols] = df[new_ratio_cols].replace([np.inf, -np.inf], np.nan)
for col in new_ratio_cols:
    df[col] = winsorize(df[col]).astype("float32")

df[new_ratio_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
fixed_assets_share,1745282.0,0.339990,0.305963,0.0,0.077924,0.237144,0.562421,0.992040
ltdb_share_of_ncli,1745282.0,0.560047,0.398275,0.0,0.024138,0.709348,0.936734,0.999793


## 2. Config

`FEATURE_COLS` is built by exclusion, so it automatically stays in sync if
new engineered columns get added upstream — but the exclusion list is now
explicit about *why* each group is dropped.

In [4]:
YEAR_COL = "closdate_year"
TARGET   = "ncliGrowthNextYear"

# Identifiers / redundant-year info -> never used as features
# 'name' and 'dateinc' already excluded at load time (see Section 1)
ID_COLS = ['idnr', 'naics_core_code']

# Raw balance-sheet levels: EXCLUDED as features.
# - Collinear with each other and with the engineered ratios built from them
#   (toas == tshf; ncli == ltdb + oncl; cuas == stok+debt+ocas+cash; etc.)
# - Contain extreme outliers (toas up to ~7.9e14 in the real data) that
#   destabilize unregularized/linear models.
# Use the engineered ratios / log_toas / log_empl instead.
RAW_LEVEL_COLS = [
    'fias', 'ifas', 'tfas', 'ofas', 'cuas', 'stok', 'debt', 'ocas', 'cash',
    'toas', 'shfd', 'capi', 'osfd', 'ncli', 'ltdb', 'oncl', 'prov', 'culi',
    'loan', 'cred', 'ocli', 'tshf', 'wkca',
]

EXCLUDE_COLS = ID_COLS + RAW_LEVEL_COLS

FEATURE_COLS = df.columns.difference([YEAR_COL, TARGET, *EXCLUDE_COLS]).tolist()
CATEGORY_COLS = ['naics_2digit', 'type']  # one-hot encoded

print(f"{len(FEATURE_COLS)} feature columns (engineered ratios + macro/sentiment + categoricals):")
print(FEATURE_COLS)

MIN_TRAIN_YEARS = 5
TEST_YEARS = [2020, 2021, 2022, 2023]

52 feature columns (engineered ratios + macro/sentiment + categoricals):
['cash_growth', 'cash_ratio', 'constraining_ratio', 'current_ratio', 'de_construction_confidence', 'de_construction_confidence_growth', 'de_consumer_confidence', 'de_consumer_confidence_growth', 'de_economic_sentiment_index', 'de_economic_sentiment_index_growth', 'de_employment_expectations_index', 'de_employment_expectations_index_growth', 'de_industry_confidence', 'de_industry_confidence_growth', 'de_retail_confidence', 'de_retail_confidence_growth', 'de_services_confidence', 'de_services_confidence_growth', 'empl', 'firm_age', 'fixed_assets_share', 'gearing', 'growth_volatility', 'ifo_business_climate', 'ifo_business_climate_growth', 'ifo_business_expectations', 'ifo_business_expectations_growth', 'ifo_business_situation', 'ifo_business_situation_growth', 'inventory_share', 'is_na_empl', 'leverage', 'litigious_ratio', 'lm_negative', 'lm_polarity', 'lm_positive', 'log_empl', 'log_toas', 'ltdb_share_of_ncli', 'na

Reasoning for `MIN_TRAIN_YEARS = 5`:
- Early years have far fewer, differently-composed firms (panel coverage
  expanded over time).
- If a CV fold trains only on those sparse years, its score reflects a
  coverage-composition shift, not real forecasting difficulty.
- `MIN_TRAIN_YEARS = 5` dilutes the sparse years to roughly ~15-20% of
  fold 1's training set (see the table below).

In [5]:
year_counts = df[YEAR_COL].value_counts().sort_index()
print(year_counts)

years_sorted = sorted(year_counts.index)
SPARSE_CUTOFF_YEAR = 2013  # adjust to wherever coverage visibly stabilizes above

print(f"\n{'min_train_years':>16} {'first_test_year':>16} {'sparse_share':>14}")
for m in range(4, 10):
    if m >= len(years_sorted):
        continue
    train_years = years_sorted[:m]
    total = sum(year_counts[y] for y in train_years)
    sparse = sum(year_counts[y] for y in train_years if y < SPARSE_CUTOFF_YEAR)
    share = sparse / total if total else float("nan")
    print(f"{m:>16} {years_sorted[m]:>16} {share:>13.1%}")

closdate_year
2010     13132
2011     12950
2012     21695
2013     39951
2014    138297
2015    145276
2016    149169
2017    190184
2018    201605
2019    201110
2020    219759
2021    225653
2022    184816
2023      1685
Name: count, dtype: int64

 min_train_years  first_test_year   sparse_share
               4             2014         54.5%
               5             2015         21.1%
               6             2016         12.9%
               7             2017          9.2%
               8             2018          6.7%
               9             2019          5.2%


**Why the test set starts at 2020 (isolating COVID on the test side only):**
this measures "if an unprecedented shock hits and the model has never seen
anything like it, how badly does it break?" — the realistic deployment
scenario for a bank planning financing programmes, which can't guarantee
the next crisis looks like anything in its training data.

## 3. Fold + metric functions

In [6]:
def rolling_origin_folds(years, min_train_years=4):
    """Expanding-window folds: train on years[:i], test on years[i].

    Example with years 2010..2019, min_train_years=5:
        Fold 1: train <=2014, test 2015
        Fold 2: train <=2015, test 2016
        ...
        Fold 5: train <=2018, test 2019
    """
    years = sorted(set(years))
    return [(years[:i], years[i]) for i in range(min_train_years, len(years))]


def regression_metrics(y_true, y_pred, baseline_value):
    """RMSE, MAE, R2_oos vs. a naive baseline (e.g. train-set mean).

    R2_oos > 0 means the model beats "always predict the baseline value".
    Use the SAME baseline_value (train-set mean) across models being compared.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]

    if len(yt) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2_oos": np.nan, "n": 0}

    err = yt - yp
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))
    ss_res = np.sum(err ** 2)
    ss_baseline = np.sum((yt - baseline_value) ** 2)
    r2_oos = float(1 - ss_res / ss_baseline) if ss_baseline > 0 else np.nan

    return {"RMSE": rmse, "MAE": mae, "R2_oos": r2_oos, "n": int(mask.sum())}

## 4. Train/test split + CV folds

- **Train/development:** 2010–2019
- **Final test:** 2020–2023 (set aside, never touched during tuning)
- **Rolling validation:** expanding-window CV folds built from the train
  set only, so test years can never leak into tuning.

In [7]:
# Replace +/-inf (from pct_change() on variables that can be zero/negative,
# e.g. de_consumer_confidence_growth, cash_growth) BEFORE splitting, so every
# fold inherits the fix.
df[FEATURE_COLS] = df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)

# Downcast numeric feature columns to float32 to roughly halve memory use
# during imputation/one-hot encoding across CV folds.
numeric_cols_all = [c for c in FEATURE_COLS if c not in CATEGORY_COLS]
df[numeric_cols_all] = df[numeric_cols_all].astype("float32")

train_df = df[df[YEAR_COL] < min(TEST_YEARS)].copy()
test_df  = df[df[YEAR_COL].isin(TEST_YEARS)].copy()
baseline_value = train_df[TARGET].mean()  # SAME baseline for every model

print(f"train: {train_df.shape}  (years {train_df[YEAR_COL].min()}-{train_df[YEAR_COL].max()})")
print(f"test:  {test_df.shape}  (years {test_df[YEAR_COL].min()}-{test_df[YEAR_COL].max()})")
print(f"baseline_value (train mean): {baseline_value:.4f}")

train: (1113369, 79)  (years 2010-2019)
test:  (631913, 79)  (years 2020-2023)
baseline_value (train mean): -0.0689


In [8]:
years = train_df[YEAR_COL].unique()
cv_folds = rolling_origin_folds(years, min_train_years=MIN_TRAIN_YEARS)

for train_years, test_year in cv_folds:
    print(f"train <= {max(train_years)} ({len(train_years)} yrs)  ->  test {test_year}")

train <= 2014 (5 yrs)  ->  test 2015
train <= 2015 (6 yrs)  ->  test 2016
train <= 2016 (7 yrs)  ->  test 2017
train <= 2017 (8 yrs)  ->  test 2018
train <= 2018 (9 yrs)  ->  test 2019


## 5. Model 1 — Lasso (regularized linear model)

Preferred over plain OLS here: the engineered ratios are correlated with
each other by construction (e.g. `leverage` and `solvency` are
near-mirrors), so Lasso's L1 penalty gives more stable coefficients and
doubles as feature selection — useful for the "model interpretation"
part of the deliverable.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = [c for c in FEATURE_COLS if c not in CATEGORY_COLS]

def make_lasso_pipeline():
    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORY_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LassoCV(cv=3, max_iter=5000, n_jobs=-1, random_state=42)),
    ])

print("Tuning Lasso across CV folds...")
for train_years, test_year in cv_folds:
    fold_train = train_df[train_df[YEAR_COL].isin(train_years)]
    fold_test  = train_df[train_df[YEAR_COL] == test_year]

    pipe = make_lasso_pipeline()
    pipe.fit(fold_train[FEATURE_COLS], fold_train[TARGET])
    preds = pipe.predict(fold_test[FEATURE_COLS])

    m = regression_metrics(fold_test[TARGET].values, preds, fold_train[TARGET].mean())
    print(test_year, m)

    del pipe
    gc.collect()

Tuning Lasso across CV folds...
2015 {'RMSE': 0.4157775318212721, 'MAE': 0.3225483549736027, 'R2_oos': -0.1500875789342071, 'n': 145276}
2016 {'RMSE': 0.49067886814448286, 'MAE': 0.4042316573094458, 'R2_oos': -0.6852919508020259, 'n': 149169}
2017 {'RMSE': 0.4144134105355459, 'MAE': 0.31586738143342463, 'R2_oos': -0.009390471381633514, 'n': 190184}
2018 {'RMSE': 0.3764832781819111, 'MAE': 0.2850166540659222, 'R2_oos': 0.057502520788958233, 'n': 201605}
2019 {'RMSE': 0.4046269332312359, 'MAE': 0.3115641621091797, 'R2_oos': -0.006353614049116718, 'n': 201110}


In [11]:
# Final refit on the full train set, scored on the held-out 2020-2023 test set
lasso_final = make_lasso_pipeline()
lasso_final.fit(train_df[FEATURE_COLS], train_df[TARGET])
lasso_preds = lasso_final.predict(test_df[FEATURE_COLS])

lasso_metrics = regression_metrics(test_df[TARGET].values, lasso_preds, baseline_value)
print("Lasso final (2020-2023 test):", lasso_metrics)
print("chosen alpha:", lasso_final.named_steps["model"].alpha_)

Lasso final (2020-2023 test): {'RMSE': 0.4052333537697719, 'MAE': 0.31631739819715454, 'R2_oos': -0.1501129124461016, 'n': 631913}
chosen alpha: 6.906215237505548e-05


In [12]:
# Interpretation: which features survived regularization
feature_names = lasso_final.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(lasso_final.named_steps["model"].coef_, index=feature_names)
nonzero = coefs[coefs != 0]
print(f"{len(nonzero)} / {len(coefs)} features survived regularization")
nonzero.reindex(nonzero.abs().sort_values(ascending=False).index).head(15).round(4)

52 / 102 features survived regularization


num__ltdb_share_of_ncli                                             -0.0658
cat__naics_2digit_55                                                -0.0614
num__leverage                                                       -0.0508
num__fixed_assets_share                                              0.0435
num__solvency                                                       -0.0391
num__ifo_business_situation                                         -0.0377
cat__naics_2digit_71                                                -0.0332
num__lm_negative                                                     0.0304
cat__type_Registered cooperative - eG                                0.0266
cat__type_Limited liability company & partnership - GmbH & Co. KG   -0.0239
num__is_na_empl                                                      0.0237
num__working_capital_ratio                                           0.0215
num__log_empl                                                        0.0208
cat__naics_2

## 6. Model 2 — XGBoost

Handles missing values and categorical columns natively — no imputation
or one-hot encoding needed, which also keeps this model's memory
footprint much lower than the Lasso pipeline's.

In [13]:
import xgboost as xgb

def prep_xgb_frame(frame, category_dtypes=None):
    out = frame[FEATURE_COLS].copy()
    for c in CATEGORY_COLS:
        if category_dtypes is not None:
            out[c] = out[c].astype("category").cat.set_categories(category_dtypes[c])
        else:
            out[c] = out[c].astype("category")
    return out

def make_xgb_model():
    return xgb.XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

print("Tuning XGBoost across CV folds...")
for train_years, test_year in cv_folds:
    fold_train = train_df[train_df[YEAR_COL].isin(train_years)]
    fold_test  = train_df[train_df[YEAR_COL] == test_year]

    X_fold_train = prep_xgb_frame(fold_train)
    cat_dtypes = {c: X_fold_train[c].cat.categories for c in CATEGORY_COLS}
    X_fold_test = prep_xgb_frame(fold_test, category_dtypes=cat_dtypes)

    model = make_xgb_model()
    model.fit(X_fold_train, fold_train[TARGET])
    preds = model.predict(X_fold_test)

    m = regression_metrics(fold_test[TARGET].values, preds, fold_train[TARGET].mean())
    print(test_year, m)

    del model, X_fold_train, X_fold_test
    gc.collect()

Tuning XGBoost across CV folds...
2015 {'RMSE': 0.3808468114836642, 'MAE': 0.2882642908681215, 'R2_oos': 0.03503951961717322, 'n': 145276}
2016 {'RMSE': 0.36247566698838823, 'MAE': 0.27144717011296576, 'R2_oos': 0.08031701381676637, 'n': 149169}
2017 {'RMSE': 0.3871186131841694, 'MAE': 0.29770163160459084, 'R2_oos': 0.11919513335823351, 'n': 190184}
2018 {'RMSE': 0.3724426412049449, 'MAE': 0.2848424866914151, 'R2_oos': 0.07762481569500856, 'n': 201605}
2019 {'RMSE': 0.38144448176916623, 'MAE': 0.28839442528375, 'R2_oos': 0.10565783064192313, 'n': 201110}


In [14]:
# Final refit on the full train set, scored on the held-out 2020-2023 test set
X_train_xgb = prep_xgb_frame(train_df)
cat_dtypes = {c: X_train_xgb[c].cat.categories for c in CATEGORY_COLS}
X_test_xgb = prep_xgb_frame(test_df, category_dtypes=cat_dtypes)

xgb_final = make_xgb_model()
xgb_final.fit(X_train_xgb, train_df[TARGET])
xgb_preds = xgb_final.predict(X_test_xgb)

xgb_metrics = regression_metrics(test_df[TARGET].values, xgb_preds, baseline_value)
print("XGBoost final (2020-2023 test):", xgb_metrics)

XGBoost final (2020-2023 test): {'RMSE': 0.37079432077801766, 'MAE': 0.27978669683055174, 'R2_oos': 0.03706658172459887, 'n': 631913}


In [15]:
importances = pd.Series(
    xgb_final.feature_importances_,
    index=xgb_final.get_booster().feature_names,
).sort_values(ascending=False)
importances.head(15).round(4)

current_ratio                              0.1646
de_construction_confidence_growth          0.0801
lm_negative                                0.0792
is_na_empl                                 0.0695
ltdb_share_of_ncli                         0.0514
lm_polarity                                0.0352
gearing                                    0.0335
fixed_assets_share                         0.0257
de_employment_expectations_index_growth    0.0238
quick_ratio                                0.0202
empl                                       0.0183
de_services_confidence_growth              0.0173
litigious_ratio                            0.0170
lm_positive                                0.0161
log_empl                                   0.0156
dtype: float32

## 7. Final model comparison (2020–2023 held-out test set)

In [16]:
naive_metrics = regression_metrics(
    test_df[TARGET].values,
    np.full(len(test_df), baseline_value),
    baseline_value,
)

comparison = pd.DataFrame({
    "Naive (train mean)": naive_metrics,
    "Lasso": lasso_metrics,
    "XGBoost": xgb_metrics,
}).T
comparison.round(4)

,RMSE,MAE,R2_oos,n
Naive (train mean),0.3779,0.2778,0.0000,631913.0
Lasso,0.4052,0.3163,-0.1501,631913.0
XGBoost,0.3708,0.2798,0.0371,631913.0


**Next steps / open items:**
- If Lasso or XGBoost meaningfully beat the naive baseline (`R2_oos > 0`),
  that's the headline result for the "expected out-of-sample performance"
  deliverable section.
- Consider adding a third model (e.g. Random Forest, as in the professor's
  slides) for a fuller comparison.
- The rolling-origin CV metrics per fold (Section 5/6 loops) are worth
  plotting — do R2_oos trend up/down as more training years become
  available? That's relevant to your research question about how
  predictive value evolves.